# LowerTE

LowerTE pass 负责将 Relay 表达式转换为 TIR PrimFunc。

测试用例覆盖了不同场景下的转换，包括原始函数、编译器函数和外部函数的处理。

In [1]:
import set_env

In [ ]:
import tvm
import tvm.testing
import logging

logging.basicConfig()
logger = logging.getLogger("test_pass_lower_te")
logger.setLevel(logging.INFO)

# 由于 TE 编译器需要良好的重构，因此它尚未作为 '标准' pass 暴露在 relay.transform 中
# 为了测试，直接获取该全局函数
LowerTE = tvm._ffi.get_global_func("relay.tec.LowerTE")

In [3]:
def transform(mod):
    """执行转换流程，应用 LowerTE pass 到给定模块

    参数:
        mod: 输入的 TVM Relay 模块

    返回:
        tvm.ir.IRModule: 应用 LowerTE pass 后的模块
    """
    logger.info("Starting module:\n%s", mod)
    # 设置主机目标和原始目标
    host_target = tvm.target.Target("llvm")
    prim_target = tvm.target.Target("llvm", host=host_target)
    # 创建转换上下文和编译配置
    ctxt = tvm.transform.PassContext()
    config = tvm.target.make_compilation_config(ctxt, prim_target)
    # 应用设备规划转换
    mod = tvm.relay.transform.PlanDevices(config)(mod)
    # 应用类型推断转换
    mod = tvm.relay.transform.InferType()(mod)
    # 应用 LowerTE 转换，将 Relay 表达式转换为 TIR PrimFunc
    mod = LowerTE("test", config)(mod)
    # 再次应用类型推断以确保类型正确
    mod = tvm.relay.transform.InferType()(mod)
    logger.info("After LowerTE:\n%s", mod)
    return mod

所有尝试使用结构等价测试与从 Relay 文本解析的预期 IRModule 进行比较的尝试都因设置正确的 call_lower 属性的难度而受挫，特别是设置具有正确 GlobalVar 实例的属性。因此，下面通过直接断言结构正确性来测试。

## 测试将原语 (Primitive) 函数转换为 TIR PrimFunc

In [4]:
# 创建包含原始函数的输入模块
input_mod = tvm.relay.parse(
    """
    #[version = "0.0.5"]
    def @main(%a: Tensor[(5, 7), float32]) -> Tensor[(5, 7), float32] {
        %0 = fn(%x : Tensor[(5, 7), float32], %y : Tensor[(5, 7), float32], Primitive=1) -> Tensor[(5, 7), float32] {
        add(%x, %y)
        };
        %0(%a, %a)
    }
    """,
    "from_string",
    None,
    None,
)

# 应用转换
actual_mod = transform(input_mod)
actual_mod.show()

INFO:test_pass_lower_te:Starting module:
def @main(%a: Tensor[(5, 7), float32] /* ty=Tensor[(5, 7), float32] span=from_string:7:16 */) -> Tensor[(5, 7), float32] {
  %0 = fn (%x: Tensor[(5, 7), float32] /* ty=Tensor[(5, 7), float32] span=from_string:5:13 */, %y: Tensor[(5, 7), float32] /* ty=Tensor[(5, 7), float32] span=from_string:5:17 */, Primitive=1) -> Tensor[(5, 7), float32] {
    add(%x, %y) /* ty=Tensor[(5, 7), float32] span=from_string:5:9 */
  } /* ty=fn (Tensor[(5, 7), float32], Tensor[(5, 7), float32]) -> Tensor[(5, 7), float32] span=from_string:7:9 */;
  %0(%a, %a) /* ty=Tensor[(5, 7), float32] span=from_string:4:9 */
}

INFO:test_pass_lower_te:After LowerTE:
def @main(%a {virtual_device=VirtualDevice(device_type=1, virtual_device_id=0, target=Target(id=5593a59e0640, kind='llvm', keys={'cpu'}, attrs={'mtriple': "x86_64-unknown-linux-gnu"}, host=Target(id=5593a5ed1a30, kind='llvm', keys={'cpu'}, attrs={'mtriple': "x86_64-unknown-linux-gnu"})))}: Tensor[(5, 7), float32] /* ty

In [6]:
# 验证主函数结构
main = actual_mod["main"]
call = main.body
assert call.op.name == "call_lowered"
assert len(call.args) == 2
assert call.args[0].name_hint == "test_fused_add"
assert len(call.args[1].fields) == 2
assert call.args[1].fields[0].name_hint == "a"
assert call.args[1].fields[1].name_hint == "a"
assert call.attrs.metadata["relay_attrs"].Primitive == 1
assert len(call.attrs.metadata["all_prim_fn_vars"]) == 1
assert call.attrs.metadata["all_prim_fn_vars"][0].name_hint == "test_fused_add"
# 验证生成的 PrimFunc
test_fused_add = actual_mod["test_fused_add"]
assert isinstance(test_fused_add, tvm.tir.PrimFunc)

## 测试将带有编译器标记的函数转换为外部函数

验证标记为 Compiler="test_pass_lower_te" 的函数能否被正确处理并转换为外部函数：

In [8]:
# 注册测试用的外部函数
@tvm._ffi.register_func("relay.ext.test_pass_lower_te")
def relay_ext_test_pass_lower_te(func):
    return None

# 创建包含编译器函数的输入模块
input_mod = tvm.relay.parse(
    """
    #[version = "0.0.5"]
    def @main(%a: Tensor[(5, 7), float32]) -> Tensor[(5, 7), float32] {
        %0 = fn(%x : Tensor[(5, 7), float32], %y : Tensor[(5, 7), float32], Primitive=1, Compiler="test_pass_lower_te", global_symbol="test_add") -> Tensor[(5, 7), float32] {
        add(%x, %y)
        };
        %0(%a, %a)
    }
    """,
    "from_string",
    None,
    None,
)

# 应用转换
actual_mod = transform(input_mod)

# 预期结果:
#   def @main(%a : Tensor[(5, 7), float32]) -> Tensor[(5, 7), float32] {
#     %0 = (%a, %a)
#     call_lowered(@test_add , %0, metadata={relay_attrs={Primitive=1, Compiler="test_pass_lower_te", global_symbol="test_add"}}, all_prim_fn_vars=[]})
#   }
#   def @test_add(%x: Tensor[(5, 7), float32], %y: Tensor[(5, 7), float32], Extern=1) -> Tensor[(5, 7), float32] {
#     add(%x, %y)
#   }

# 验证主函数结构
main = actual_mod["main"]
call = main.body
assert call.op.name == "call_lowered"
assert len(call.args) == 2
assert call.args[0].name_hint == "test_add"
assert len(call.args[1].fields) == 2
assert call.args[1].fields[0].name_hint == "a"
assert call.args[1].fields[1].name_hint == "a"
assert call.attrs.metadata["relay_attrs"].Primitive == 1
assert call.attrs.metadata["relay_attrs"].Compiler == "test_pass_lower_te"
assert call.attrs.metadata["relay_attrs"].global_symbol == "test_add"
assert len(call.attrs.metadata["all_prim_fn_vars"]) == 0

# 验证生成的外部函数
test_add = actual_mod["test_add"]
assert isinstance(test_add, tvm.relay.Function)
assert test_add.attrs["Extern"] == 1

INFO:test_pass_lower_te:Starting module:
def @main(%a: Tensor[(5, 7), float32] /* ty=Tensor[(5, 7), float32] span=from_string:7:16 */) -> Tensor[(5, 7), float32] {
  %0 = fn (%x: Tensor[(5, 7), float32] /* ty=Tensor[(5, 7), float32] span=from_string:5:13 */, %y: Tensor[(5, 7), float32] /* ty=Tensor[(5, 7), float32] span=from_string:5:17 */, Primitive=1, Compiler="test_pass_lower_te", global_symbol="test_add") -> Tensor[(5, 7), float32] {
    add(%x, %y) /* ty=Tensor[(5, 7), float32] span=from_string:5:9 */
  } /* ty=fn (Tensor[(5, 7), float32], Tensor[(5, 7), float32]) -> Tensor[(5, 7), float32] span=from_string:7:9 */;
  %0(%a, %a) /* ty=Tensor[(5, 7), float32] span=from_string:4:9 */
}

INFO:test_pass_lower_te:After LowerTE:
def @main(%a {virtual_device=VirtualDevice(device_type=1, virtual_device_id=0, target=Target(id=5593a5a0b1f0, kind='llvm', keys={'cpu'}, attrs={'mtriple': "x86_64-unknown-linux-gnu"}, host=Target(id=5593a5bd4bf0, kind='llvm', keys={'cpu'}, attrs={'mtriple': "x86_

## 测试处理已标记为外部的函数
验证已经标记为 Extern=1 的全局函数能否被正确处理：

In [9]:
# 创建包含外部函数的输入模块
input_mod = tvm.relay.parse(
    """
    #[version = "0.0.5"]
    def @main(%a: Tensor[(5, 7), float32]) -> Tensor[(5, 7), float32] {
        @my_add(%a, %a)
    }
    def @my_add(%x : Tensor[(5, 7), float32], %y : Tensor[(5, 7), float32], Extern=1) -> Tensor[(5, 7), float32] {
        add(%x, %y)
    }
    """,
    "from_string",
    None,
    None,
)

# 应用转换
actual_mod = transform(input_mod)

# 预期结果:
#   def @main(%a: Tensor[(5, 7), float32]) -> Tensor[(5, 7), float32] {
#     %0 = (%a, %a);
#     call_lowered(@my_add, %0, metadata={relay_attrs={Extern=1}}, all_prim_fn_vars=[]})
#   }
#   def @my_add(%x: Tensor[(5, 7), float32], %y: Tensor[(5, 7), float32], Extern=1) -> Tensor[(5, 7), float32] {
#     add(%x, %y)
#   }

# 验证主函数结构
main = actual_mod["main"]
call = main.body
assert call.op.name == "call_lowered"
assert len(call.args) == 2
assert call.args[0].name_hint == "my_add"
assert len(call.args[1].fields) == 2
assert call.args[1].fields[0].name_hint == "a"
assert call.args[1].fields[1].name_hint == "a"
assert call.attrs.metadata["relay_attrs"].Extern == 1
assert len(call.attrs.metadata["all_prim_fn_vars"]) == 0

# 验证外部函数保持不变
test_add = actual_mod["my_add"]
assert isinstance(test_add, tvm.relay.Function)
assert test_add.attrs["Extern"] == 1

INFO:test_pass_lower_te:Starting module:
def @main(%a: Tensor[(5, 7), float32] /* ty=Tensor[(5, 7), float32] span=from_string:4:22 */) -> Tensor[(5, 7), float32] {
  @my_add(%a, %a) /* ty=Tensor[(5, 7), float32] span=from_string:4:9 */
}

def @my_add(%x: Tensor[(5, 7), float32] /* ty=Tensor[(5, 7), float32] span=from_string:7:13 */, %y: Tensor[(5, 7), float32] /* ty=Tensor[(5, 7), float32] span=from_string:7:17 */, Extern=1) -> Tensor[(5, 7), float32] {
  add(%x, %y) /* ty=Tensor[(5, 7), float32] span=from_string:7:9 */
}

INFO:test_pass_lower_te:After LowerTE:
def @main(%a {virtual_device=VirtualDevice(device_type=1, virtual_device_id=0, target=Target(id=5593a574e8c0, kind='llvm', keys={'cpu'}, attrs={'mtriple': "x86_64-unknown-linux-gnu"}, host=Target(id=5593a5725150, kind='llvm', keys={'cpu'}, attrs={'mtriple': "x86_64-unknown-linux-gnu"})))}: Tensor[(5, 7), float32] /* ty=Tensor[(5, 7), float32] span=from_string:4:22 */, virtual_device=VirtualDevice(device_type=1, virtual_device_id

## 测试处理具有动态形状的外部函数

验证带有动态形状（`?` 维度）的外部函数能否被正确处理，并生成相应的形状函数

In [10]:
# 创建包含动态形状外部函数的输入模块
input_mod = tvm.relay.parse(
    """
    #[version = "0.0.5"]
    def @main(%a: Tensor[(5, 7), float32]) -> Tensor[(?, ?), float32] {
        @my_dyn(%a, %a)
    }
    def @my_dyn(%x : Tensor[(5, 7), float32], %y : Tensor[(5, 7), float32], Extern=1) -> Tensor[(?, ?), float32] {
        add(%x, %y)
    }
    """,
    "from_string",
    None,
    None,
)

# 应用转换
actual_mod = transform(input_mod)

# 预期结果:
# def @main(%a: Tensor[(5, 7), float32]) -> Tensor[(?, ?), float32] {
#   %0 = (%a, %a);
#   call_lowered(@my_dyn, %0, metadata={prim_shape_fn_var='test_shape_func_add', relay_attrs={Extern=1}, prim_shape_fn_states=[2, 2], prim_shape_fn_num_inputs=2, all_prim_shape_fn_vars=['shape_func_add'], prim_shape_fn_num_outputs=1, all_prim_fn_vars=[]})
# }
# def @my_dyn(%x: Tensor[(5, 7), float32] , %y: Tensor[(5, 7), float32] , Extern=1) -> Tensor[(?, ?), float32] {
#   add(%x, %y)
# }
# def @test_shape_func_add = <shape PrimFunc>

# 验证主函数结构和动态形状相关的元数据
main = actual_mod["main"]
call = main.body
assert call.op.name == "call_lowered"
assert len(call.args) == 2
assert call.args[0].name_hint == "my_dyn"
assert len(call.args[1].fields) == 2
assert call.args[1].fields[0].name_hint == "a"
assert call.args[1].fields[1].name_hint == "a"
assert call.attrs.metadata["prim_shape_fn_var"].name_hint == "test_shape_func_add"
assert call.attrs.metadata["relay_attrs"].Extern == 1
assert len(call.attrs.metadata["prim_shape_fn_states"]) == 2
assert call.attrs.metadata["prim_shape_fn_states"][0] == 2
assert call.attrs.metadata["prim_shape_fn_states"][1] == 2
assert call.attrs.metadata["prim_shape_fn_num_inputs"] == 2
assert len(call.attrs.metadata["all_prim_shape_fn_vars"]) == 1
assert call.attrs.metadata["all_prim_shape_fn_vars"][0].name_hint == "test_shape_func_add"
assert call.attrs.metadata["prim_shape_fn_num_outputs"] == 1
assert len(call.attrs.metadata["all_prim_fn_vars"]) == 0

# 验证外部函数保持不变
my_dyn = actual_mod["my_dyn"]
assert isinstance(my_dyn, tvm.relay.Function)
assert my_dyn.attrs["Extern"] == 1

# 验证生成的形状函数
shape_func_add = actual_mod["test_shape_func_add"]
assert isinstance(shape_func_add, tvm.tir.PrimFunc)

INFO:test_pass_lower_te:Starting module:
def @main(%a: Tensor[(5, 7), float32] /* ty=Tensor[(5, 7), float32] span=from_string:4:22 */) -> Tensor[(?, ?), float32] {
  @my_dyn(%a, %a) /* ty=Tensor[(?, ?), float32] span=from_string:4:9 */
}

def @my_dyn(%x: Tensor[(5, 7), float32] /* ty=Tensor[(5, 7), float32] span=from_string:7:13 */, %y: Tensor[(5, 7), float32] /* ty=Tensor[(5, 7), float32] span=from_string:7:17 */, Extern=1) -> Tensor[(?, ?), float32] {
  add(%x, %y) /* ty=Tensor[(?, ?), float32] span=from_string:7:9 */
}

INFO:test_pass_lower_te:After LowerTE:
def @main(%a {virtual_device=VirtualDevice(device_type=1, virtual_device_id=0, target=Target(id=5593a598f300, kind='llvm', keys={'cpu'}, attrs={'mtriple': "x86_64-unknown-linux-gnu"}, host=Target(id=5593a595af10, kind='llvm', keys={'cpu'}, attrs={'mtriple': "x86_64-unknown-linux-gnu"})))}: Tensor[(5, 7), float32] /* ty=Tensor[(5, 7), float32] span=from_string:4:22 */, virtual_device=VirtualDevice(device_type=1, virtual_device_id